# Divar Real Estate Analysis — Recommender System (Clustering)

**Problem 1:** Clustering properties to support a property recommender system

**Author:** Mahdi Tarrah

**Data:** `data/Divar.csv`

In [1]:
#libraries
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import utm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
clustering_columns = ['price_value','building_size','rooms_count','location_latitude','location_longitude','cat2_slug','cat3_slug','city_slug']
df = pd.read_csv('../data/Divar.csv', usecols=clustering_columns)
print(df.shape)
df.head()

(1000000, 8)


,cat2_slug,cat3_slug,city_slug,price_value,building_size,rooms_count,location_latitude,location_longitude
0,temporary-rent,villa,karaj,NaN,500.0,سه,35.811684,50.936600
1,residential-sell,apartment-sell,tehran,8.500000e+09,60.0,یک,NaN,NaN
2,residential-rent,apartment-rent,tehran,NaN,132.0,سه,35.703865,51.373459
3,commercial-rent,office-rent,tehran,NaN,90.0,یک,NaN,NaN
4,residential-sell,apartment-sell,mashhad,5.750000e+09,115.0,دو,NaN,NaN


In [3]:
#filled
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   cat2_slug           1000000 non-null  str    
 1   cat3_slug           999999 non-null   str    
 2   city_slug           999998 non-null   str    
 3   price_value         568346 non-null   float64
 4   building_size       980394 non-null   float64
 5   rooms_count         845899 non-null   str    
 6   location_latitude   655608 non-null   float64
 7   location_longitude  655608 non-null   float64
dtypes: float64(4), str(4)
memory usage: 61.0 MB


In [4]:
#copy
data = df.copy()

#clean
data.loc[data['price_value'] <= 0, 'price_value'] = np.nan

def clean_outliers(data,column):
    """Set outliers to the NaN."""
    logs = np.log(data[column])
    q1, q3 = logs.quantile(0.25),logs.quantile(0.75)
    iqr = q3 - q1
    data.loc[~logs.between(q1 - 1.5 * iqr,q3 + 1.5 * iqr), column] = np.nan
    return data

data = clean_outliers(data,'price_value')
data = clean_outliers(data,'building_size')

inside_iran = (data['location_latitude'].between(25,40)) & (data['location_longitude'].between(44,63.5))
data.loc[~inside_iran,['location_latitude','location_longitude']] = np.nan

data = data.dropna(subset=['building_size','price_value','location_latitude','location_longitude'])
print(data.shape)
print(data[['price_value','building_size']].describe())

(324485, 8)
        price_value  building_size
count  3.244850e+05  324485.000000
mean   5.047053e+09     130.622852
std    6.172471e+09      87.705984
min    1.640000e+08      23.000000
25%    1.600000e+09      75.000000
50%    3.000000e+09     100.000000
75%    5.850000e+09     153.000000
max    5.100000e+10     538.000000


In [5]:
xy = [utm.from_latlon(lat, lon, force_zone_number=39)[:2] 
for lat, lon in zip(data['location_latitude'], data['location_longitude'])]

data['utm_x'] = [p[0] for p in xy]
data['utm_y'] = [p[1] for p in xy]

print(data[['utm_x','utm_y']].describe())

              utm_x         utm_y
count  3.244850e+05  3.244850e+05
mean   5.549497e+05  3.888984e+06
std    2.790465e+05  2.525020e+05
min   -1.146343e+05  2.803746e+06
25%    4.763461e+05  3.860848e+06
50%    5.304762e+05  3.953651e+06
75%    5.738981e+05  4.042502e+06
max    1.660176e+06  4.407134e+06


In [ ]:
#cheak errors
print(((data['utm_x'] < 100000) | (data['utm_x'] > 900000)).sum())

51619
